# 00 — Data Preparation

Loads `data/metadata/main_metadata.csv`, parses dates, standardizes water types, joins measurement row counts, and saves `data/analysis/processed_metadata.parquet` for use by all downstream inventory notebooks.

Run this notebook first before any other notebook in `code/03_inventory/`.

In [2]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

root = Path('../..')
metadata_path  = root / 'data/metadata/main_metadata.csv'
counts_path    = root / 'data/analysis/csv_row_counts.csv'
output_path    = root / 'data/analysis/processed_metadata.parquet'

df = pd.read_csv(metadata_path, low_memory=False)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

Loaded 109,440 rows, 73 columns


## Parse dates

`dates_of_recording` is a free-text field with formats like `"01/1956"`, `"1945-1962"`, `"1923"`, `"January 1923"`. We extract the earliest and latest 4-digit year within 1800–1980.

In [3]:
OPEN_ENDED = ('present', 'current', 'ongoing', 'to date', 'today')

def parse_year_range(date_str):
    if pd.isna(date_str):
        return None, None
    s = str(date_str).lower()
    if any(t in s for t in OPEN_ENDED):
        return None, None
    years = [int(y) for y in re.findall(r'\b(1[89]\d{2}|20[0-7]\d)\b', s)
             if 1800 <= int(y) <= 1980]
    if not years:
        return None, None
    return min(years), max(years)

year_pairs = df['dates_of_recording'].map(parse_year_range)
df['year_start'] = year_pairs.map(lambda x: x[0]).astype('Int64')
df['year_end']   = year_pairs.map(lambda x: x[1]).astype('Int64')
df['decade_start'] = (df['year_start'] // 10 * 10).astype('Int64')

missing = df['year_start'].isna().sum()
print(f'Parsed dates: {len(df) - missing:,} rows have years, {missing:,} missing ({missing/len(df):.1%})')

Parsed dates: 88,077 rows have years, 21,363 missing (19.5%)


## Standardize water types

In [4]:
WATER_TYPE_MAP = {
    'stream discharge': 'Stream Discharge',
    'groundwater':      'Groundwater',
    'reservoir':        'Reservoir',
    'irrigation':       'Irrigation',
    'springs':          'Springs',
    'precipitation':    'Precipitation',
    'water quality':    'Water Quality',
    'not water related':'Not Water Related',
    'other':            'Other',
}

def clean_water_type(wt):
    if pd.isna(wt):
        return 'Other'
    wt_lower = str(wt).strip().lower()
    for key, val in WATER_TYPE_MAP.items():
        if key in wt_lower:
            return val
    return 'Other'

df['water_type_clean'] = df['water_type'].map(clean_water_type)
print(df['water_type_clean'].value_counts())

water_type_clean
Stream Discharge     76060
Groundwater          20590
Reservoir             4384
Springs               2782
Irrigation            2383
Other                 1228
Precipitation          908
Not Water Related      758
Water Quality          347
Name: count, dtype: int64


## Build combined coordinates

Use `actual_latitude/longitude` when present (extracted directly from document), otherwise fall back to `inferred_latitude/longitude` (LLM-estimated from place names).

In [5]:
df['lat_combined'] = df['actual_latitude'].combine_first(df['inferred_latitude'])
df['lon_combined'] = df['actual_longitude'].combine_first(df['inferred_longitude'])
df['coord_source'] = np.where(
    df['actual_latitude'].notna(), 'actual',
    np.where(df['inferred_latitude'].notna(), 'inferred', None)
)

has_coords = df['lat_combined'].notna().sum()
print(f'{has_coords:,} rows have coordinates ({has_coords/len(df):.1%})')
print(df['coord_source'].value_counts(dropna=False))

101,443 rows have coordinates (92.7%)
coord_source
actual      57849
inferred    43594
None         7997
Name: count, dtype: int64


## Join measurement row counts

`csv_row_counts.csv` has one row per CSV table file. Aggregate to page level (sum across tables on the same page), then join to the metadata.

In [6]:
counts = pd.read_csv(counts_path)
print('Row counts columns:', counts.columns.tolist())

# Aggregate to page level
counts_page = (
    counts.groupby(['doc_id', 'page_number'])['number_rows']
    .sum()
    .reset_index()
    .rename(columns={'doc_id': 'id', 'number_rows': 'n_measurements'})
)

df = df.merge(counts_page, on=['id', 'page_number'], how='left')
matched = df['n_measurements'].notna().sum()
print(f'Matched row counts for {matched:,} of {len(df):,} rows')

Row counts columns: ['doc_id', 'page_number', 'table_number', 'number_rows']
Matched row counts for 106,734 of 109,440 rows


## Save

In [7]:
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(output_path, index=False)
print(f'Saved {len(df):,} rows to {output_path}')
print('\nColumn summary:')
print(df[['id','page_number','year_start','year_end','decade_start',
          'water_type_clean','lat_combined','lon_combined',
          'coord_source','n_measurements']].describe(include='all'))

Saved 109,440 rows to ../../data/analysis/processed_metadata.parquet

Column summary:
                  id    page_number   year_start     year_end  decade_start  \
count   1.094390e+05  109432.000000      88077.0      88077.0       88077.0   
unique           NaN            NaN         <NA>         <NA>          <NA>   
top              NaN            NaN         <NA>         <NA>          <NA>   
freq             NaN            NaN         <NA>         <NA>          <NA>   
mean    9.554302e+06     218.405905  1941.989271  1949.833657   1937.606526   
std     2.402125e+07     180.473698     25.56624    22.828375     25.447922   
min     1.500000e+01       1.000000       1805.0       1805.0        1800.0   
25%     1.595000e+03      63.000000       1920.0       1933.0        1920.0   
50%     3.090000e+03     172.000000       1947.0       1957.0        1940.0   
75%     5.636300e+04     340.000000       1965.0       1968.0        1960.0   
max     7.023307e+07     959.000000       198